In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from datetime import date, datetime, timedelta
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import fcluster
from matplotlib.colors import TwoSlopeNorm
sns.set_theme(style="ticks")
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Arial'
plt.rcParams['mathtext.it'] = 'Arial:italic'

# parameters

In [ ]:
yeari, yearf = '2024', '2024'
weeki, weekf = '18', '31'

In [ ]:
di = datetime.strptime(f'{yeari}-{weeki}-1', "%Y-%W-%w").date()
df = datetime.strptime(f'{yearf}-{weekf}-1', "%Y-%W-%w").date() + timedelta(6)
ds = [di+timedelta(dt) for dt in range((df-di).days+1)]
daylist = ds
print(di, 'until', df)

In [ ]:
cdef = 'tl7_10m'# 'tl5_10m' 'tl6_10m' 'tl7_10m' 'tl8_10m' 'tl8_60m'
cdef_alt = '16m_10min'# tl5: 62 ... tl7: 16   tl8: 8

# load data

In [ ]:
data = pd.read_csv('data/fig6/poi_contacts2.csv', low_memory=False)# only nodes
#data = pd.read_csv('data/fig6/poi_contacts3.csv', low_memory=False)# incl. polygons
data['day'] = [d.date() for d in pd.to_datetime(data.day)]
data['stime'] = pd.to_datetime(data.stime)
data['hour'] = data.stime.dt.hour
data

# analyses

In [ ]:
cntthresh = 3000
#mtypes = data.groupby('venue').pair.apply(len).reset_index().sort_values('pair', ascending=False)
mtypes = data.groupby(['venue','day','hour']).pair.apply(lambda x: len(set(x))).reset_index()\
             .groupby('venue').pair.sum().reset_index().sort_values('pair', ascending=False)
mtypes_big = mtypes[mtypes.pair >= cntthresh].venue.tolist()
print(len(mtypes_big), len(mtypes))
for _, row in mtypes.iterrows():
    print(row.venue, row.pair)

In [ ]:
mtypes[mtypes.venue=='*amenity:theatre']

In [ ]:
to_excl = [mt for mt in mtypes_big if mt.split(':')[0]=='boundary']
mtypes_big = [mt for mt in mtypes_big if mt not in to_excl]
print(len(mtypes_big), len(mtypes))

In [ ]:
for_heatmap = data[data.venue.isin(mtypes_big)].groupby(['venue','day','hour']).pair.apply(lambda x: len(set(x))).reset_index()
c1 = pd.DataFrame(set(for_heatmap.venue), columns=['venue'])
c2 = pd.DataFrame([d.date() for d in pd.date_range(di, df)], columns=['day'])
c3 = pd.DataFrame(list(range(24)), columns=['hour'])
for_heatmap = c1.merge(c2, how='cross').merge(c3, how='cross').merge(for_heatmap, on=['venue','day','hour'], how='left')
for_heatmap

In [ ]:
sns.set_theme(style="ticks")

for venue in mtypes_big[:]:
    
    fig, ax = plt.subplots(figsize=[10,3])
    
    pivoted = for_heatmap[for_heatmap.venue==venue].fillna(0).pivot(index="hour", columns="day", values="pair")  # Reshape
    g = sns.heatmap(pivoted.loc[::-1], cbar=False, robust=True, ax=ax, square=True, cmap='coolwarm')#, norm=norm, 
    
    # Set yticks for all subplots
    ncnt = pivoted.sum().sum()
    g.set(title=venue+f' | $n=${int(ncnt)}', yticks=[24-x-.5 for x in range(0,24,2)], xticks=[x+.5 for x in range(len(set(data.day)))])
    g.set_yticklabels(list(range(0,24,2)), rotation=0)
    g.set_xticklabels([str(d.month).zfill(2)+'/'+str(d.day).zfill(2) if d.weekday()==6 else '' for d in sorted(set(data.day))], rotation=0)
    
    plt.tight_layout()
    plt.show()

In [ ]:
sns.set_theme(style="ticks")

# Function to plot a heatmap inside each facet
def heatmap_func(data, **kwargs):
    pivoted = data.pivot(index="hour", columns="day", values="pair")  # Reshape
    sns.heatmap(pivoted.loc[::-1], cmap="coolwarm", cbar=False, robust=True, square=True, **kwargs)

npan = 18
for idxcnt, (idxi, idxf) in enumerate(zip(range(0,len(mtypes_big),npan), range(npan,len(mtypes_big)+npan,npan))):
    mtypes_here = mtypes_big[idxi:idxf]

    # Create a FacetGrid, grouping by 'category'
    g = sns.FacetGrid(for_heatmap[for_heatmap.venue.isin(mtypes_here)].reset_index().fillna(0),
                      col="venue", margin_titles=True, height=2.5, aspect=3.6, col_wrap=2,
                      col_order=mtypes_here)#['dining','outing','fanzone','shopping'])
    
    # Map the custom heatmap function to each facet
    g.map_dataframe(heatmap_func)
    
    # Set yticks for all subplots
    g.set(yticks=[24-x-.5 for x in range(0,24,2)], xticks=[x+.5 for x in range(len(set(data.day)))])
    g.set_yticklabels(list(range(0,24,2)), rotation=0)
    g.set_xticklabels([str(d.month).zfill(2)+'/'+str(d.day).zfill(2) if d.weekday()==6 else '' for d in sorted(set(data.day))], rotation=0)
    
    for ax, venue in zip(g.axes.flat, mtypes_here):
        ncnt = for_heatmap[for_heatmap.venue==venue].fillna(0).pivot(index="hour", columns="day", values="pair").sum().sum()
        ax.set_title(venue+f' | $n=${int(ncnt)}')
        ax.tick_params(axis="x", labelbottom=True)
        ax.tick_params(axis="y", labelleft=True)
    
    # Modify margin titles
    #g.set_titles(row_template="{row_name}")#, col_template="{col_name}", size=14, fontweight='bold')
    
    g.tight_layout()
    plt.subplots_adjust(wspace=-0.2)
    plt.savefig(f'plots/figs11/figs11_contacts_nationwide_poi_heatmap_{idxcnt+1}.jpg', bbox_inches='tight', dpi=300)
    plt.savefig(f'plots/figs11/figs11_contacts_nationwide_poi_heatmap_{idxcnt+1}.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
dheres = [date(2024,7,5)]

for_to_cluster2 = []
for_base, for_here = [], []
for vtype in mtypes_big[:]:
    
    dist_base = for_heatmap[(for_heatmap.venue==vtype) & (for_heatmap.day>=date(2024,6,10)) & (~for_heatmap.day.isin(dheres))].copy(deep=True)
    dist_base['wd'] = [d.weekday() for d in dist_base.day]
    dist_base = dist_base.fillna(0).groupby(['wd','hour']).pair.mean().reset_index().rename(columns={'pair':'pair_avg'})

    for_heatmap3 = for_heatmap[(for_heatmap.venue==vtype)].copy(deep=True)
    for_heatmap3['wd'] = [d.weekday() for d in for_heatmap3.day]
    for_heatmap3 = for_heatmap3.merge(dist_base, on=['wd','hour']).fillna(0)
    for_heatmap3['diff'] = (for_heatmap3.pair - for_heatmap3.pair_avg)# / for_heatmap3.pair_avg#.abs()

    for_to_cluster2.append(for_heatmap3[for_heatmap3.day.isin(dheres)].reset_index().rename(columns={'diff':vtype}).sort_values('hour')[vtype])
    for_base.append(for_heatmap3[for_heatmap3.day.isin(dheres)].reset_index().rename(columns={'pair_avg':vtype}).sort_values('hour')[vtype])
    for_here.append(for_heatmap3[for_heatmap3.day.isin(dheres)].reset_index().rename(columns={'pair':vtype}).sort_values('hour')[vtype])

to_cluster2, base, here = pd.concat(for_to_cluster2, axis=1), pd.concat(for_base, axis=1), pd.concat(for_here, axis=1)
to_cluster2.index.name, base.index.name, here.index.name = 'hour', 'hour', 'hour'
to_cluster2

In [ ]:
# normalize the vectors
scaler = StandardScaler()
normalized = scaler.fit_transform(to_cluster2)
to_cluster3 = pd.DataFrame(normalized, index=to_cluster2.index, columns=to_cluster2.columns)
to_cluster3

In [ ]:
# Draw the full plot
sns.set_theme(style="ticks")
g = sns.clustermap(to_cluster3.corr(), center=0, cmap="vlag",
                   method='ward',# 'average','single','complete','weighted','centroid','median','ward'
                   metric='euclidean',# 'euclidean','cityblock','cosine','correlation','hamming','jaccard','chebyshev','canberra','braycurtis','matching'
                   #row_colors=row_cols, col_colors=row_cols,
                   dendrogram_ratio=(.1, .2),
                   cbar_pos=(.02, .32, .03, .2),
                   linewidths=0., figsize=(12, 13))

g.ax_row_dendrogram.remove()

In [ ]:
# get clusters at desired dendrogram depth
depth = 3# no. of clusters
linkagemat = g.dendrogram_row.linkage
cluster_labels = fcluster(linkagemat, t=depth, criterion='maxclust')# Map the cluster labels to the original row labels
row_labels = pd.DataFrame(pd.Series(cluster_labels, index=to_cluster3.corr().index), columns=['cluster'])
#row_cats = pd.DataFrame(row_cols).rename(columns={'day':'category'})
cats_clus = row_labels.reset_index(names='venue')#.join(row_cats).rename(columns={'cat_label':'color'}).reset_index()
cats_clus

In [ ]:
for cl_h in range(1,1+depth):
    print(cl_h)
    mtypes_h = cats_clus[cats_clus.cluster==cl_h].venue.tolist()
    print(' '.join(sorted([str(y).split(':')[1] if ':' in str(y) else str(y) for y in mtypes_h])))
    print()

In [ ]:
weights = for_heatmap[for_heatmap.day.isin(dheres)].groupby(['venue','day']).pair.sum().reset_index()
for_markdown = []
for cl_h in range(1,1+depth):
    print('cluster', cl_h)
    mtypes_h = cats_clus[cats_clus.cluster==cl_h].venue.tolist()
    for _, row2 in weights[weights.venue.isin(mtypes_h)].sort_values('pair', ascending=False).iterrows():
        print(row2.venue, row2.pair)
        for_markdown.append([cl_h, row2.venue, row2.pair])
    print()

In [ ]:
df_markdown = pd.DataFrame(for_markdown, columns=['cluster','map feature','contacts'])
print(df_markdown.to_markdown())

|     |   cluster | map feature               |   contacts |
|----:|----------:|:--------------------------|-----------:|
|   0 |         1 | *amenity:bench            |       1002 |
|   1 |         1 | *amenity:vending_machine  |        929 |
|   2 |         1 | *amenity:bicycle_parking  |        888 |
|   3 |         1 | *tourism:information      |        768 |
|   4 |         1 | *amenity:telephone        |        639 |
|   5 |         1 | *amenity:parking          |        565 |
|   6 |         1 | *tourism:artwork          |        369 |
|   7 |         1 | *amenity:pub              |        321 |
|   8 |         1 | *amenity:fountain         |        319 |
|   9 |         1 | *amenity:ice_cream        |        310 |
|  10 |         1 | fanzone                   |        237 |
|  11 |         1 | *amenity:bar              |        220 |
|  12 |         1 | *shop:yes                 |        153 |
|  13 |         1 | *amenity:drinking_water   |        127 |
|  14 |         1 | *amenity:marketplace      |         71 |
|  15 |         1 | *amenity:biergarten       |         65 |
|  16 |         1 | *amenity:nightclub        |         51 |
|  17 |         1 | *shop:massage             |         48 |
|  18 |         2 | *shop:bakery              |       1348 |
|  19 |         2 | *shop:supermarket         |       1155 |
|  20 |         2 | *amenity:fast_food        |       1126 |
|  21 |         2 | *shop:clothes             |       1037 |
|  22 |         2 | *amenity:pharmacy         |        941 |
|  23 |         2 | *amenity:toilets          |        825 |
|  24 |         2 | *shop:hairdresser         |        803 |
|  25 |         2 | *shop:shoes               |        614 |
|  26 |         2 | *shop:chemist             |        609 |
|  27 |         2 | *shop:optician            |        454 |
|  28 |         2 | *amenity:bank             |        447 |
|  29 |         2 | *shop:florist             |        411 |
|  30 |         2 | *amenity:post_office      |        385 |
|  31 |         2 | *shop:mobile_phone        |        376 |
|  32 |         2 | *shop:butcher             |        371 |
|  33 |         2 | *shop:travel_agency       |        336 |
|  34 |         2 | *shop:electronics         |        310 |
|  35 |         2 | *shop:books               |        310 |
|  36 |         2 | *shop:jewelry             |        302 |
|  37 |         2 | *shop:beauty              |        283 |
|  38 |         2 | *amenity:charging_station |        256 |
|  39 |         2 | *shop:sports              |        248 |
|  40 |         2 | *shop:gift                |        226 |
|  41 |         2 | *shop:stationery          |        220 |
|  42 |         2 | *amenity:parking_entrance |        206 |
|  43 |         2 | *shop:newsagent           |        195 |
|  44 |         2 | *shop:perfumery           |        182 |
|  45 |         2 | *shop:toys                |        175 |
|  46 |         2 | *amenity:dentist          |        162 |
|  47 |         2 | *shop:confectionery       |        162 |
|  48 |         2 | *shop:dry_cleaning        |        123 |
|  49 |         2 | *shop:pet                 |        117 |
|  50 |         2 | *shop:vacant              |        108 |
|  51 |         2 | *shop:cosmetics           |         98 |
|  52 |         2 | *shop:car                 |         93 |
|  53 |         2 | *shop:doityourself        |         86 |
|  54 |         2 | *amenity:car_rental       |         85 |
|  55 |         2 | *shop:greengrocer         |         77 |
|  56 |         2 | *shop:tea                 |         72 |
|  57 |         2 | *shop:video_games         |         72 |
|  58 |         2 | *shop:bag                 |         70 |
|  59 |         2 | *shop:outdoor             |         62 |
|  60 |         2 | *shop:fashion_accessories |         35 |
|  61 |         2 | *shop:alcohol             |         31 |
|  62 |         2 | *amenity:bureau_de_change |         23 |
|  63 |         2 | *tourism:guest_house      |         23 |
|  64 |         3 | *amenity:restaurant       |       1449 |
|  65 |         3 | *amenity:atm              |       1077 |
|  66 |         3 | *amenity:cafe             |        867 |
|  67 |         3 | *amenity:post_box         |        841 |
|  68 |         3 | *amenity:waste_basket     |        712 |
|  69 |         3 | *tourism:viewpoint        |        610 |
|  70 |         3 | *amenity:fuel             |        506 |
|  71 |         3 | *shop:kiosk               |        443 |
|  72 |         3 | *amenity:doctors          |        363 |
|  73 |         3 | *amenity:recycling        |        339 |
|  74 |         3 | *tourism:hotel            |        236 |
|  75 |         3 | *shop:convenience         |        227 |
|  76 |         3 | *amenity:clock            |        199 |
|  77 |         3 | *shop:variety_store       |        194 |
|  78 |         3 | *shop:beverages           |        169 |
|  79 |         3 | *amenity:taxi             |        159 |
|  80 |         3 | *amenity:car_wash         |        154 |
|  81 |         3 | *shop:deli                |        152 |
|  82 |         3 | *shop:bicycle             |        147 |
|  83 |         3 | *amenity:photo_booth      |        145 |
|  84 |         3 | *shop:department_store    |        135 |
|  85 |         3 | *shop:furniture           |        130 |
|  86 |         3 | *shop:tobacco             |        121 |
|  87 |         3 | *shop:car_repair          |        115 |
|  88 |         3 | *amenity:bicycle_rental   |        111 |
|  89 |         3 | *tourism:attraction       |        106 |
|  90 |         3 | *shop:tailor              |        103 |
|  91 |         3 | *shop:interior_decoration |        101 |
|  92 |         3 | *shop:coffee              |         96 |
|  93 |         3 | *shop:photo               |         93 |
|  94 |         3 | *amenity:driving_school   |         92 |
|  95 |         3 | *shop:ticket              |         92 |
|  96 |         3 | *amenity:kindergarten     |         89 |
|  97 |         3 | *amenity:library          |         89 |
|  98 |         3 | *amenity:compressed_air   |         86 |
|  99 |         3 | *amenity:social_facility  |         84 |
| 100 |         3 | *shop:medical_supply      |         76 |
| 101 |         3 | *tourism:museum           |         76 |
| 102 |         3 | *amenity:school           |         72 |
| 103 |         3 | *amenity:place_of_worship |         70 |
| 104 |         3 | *shop:hearing_aids        |         68 |
| 105 |         3 | *amenity:police           |         61 |
| 106 |         3 | *amenity:theatre          |         59 |
| 107 |         3 | *shop:houseware           |         56 |
| 108 |         3 | *shop:boutique            |         56 |
| 109 |         3 | *shop:laundry             |         55 |
| 110 |         3 | *shop:computer            |         53 |
| 111 |         3 | *shop:erotic              |         47 |
| 112 |         3 | *amenity:grit_bin         |         46 |
| 113 |         3 | *amenity:cinema           |         44 |
| 114 |         3 | *shop:locksmith           |         41 |
| 115 |         3 | *amenity:community_centre |         38 |
| 116 |         3 | *shop:bed                 |         37 |
| 117 |         3 | *amenity:food_court       |         33 |
| 118 |         3 | *amenity:shelter          |         33 |
| 119 |         3 | *amenity:casino           |         22 |
| 120 |         3 | *shop:art                 |         20 |

In [ ]:
cats = [cats_clus[cats_clus.venue==s].cluster.iloc[0] for s in to_cluster3.columns.tolist()]
cl_labels = ['socializing', 'shopping', 'routine']
#cl_labels = ['shop','outdoor','routine']
clist = sns.husl_palette(depth, s=1.)
cat2col = dict(zip(list(set(cats)), clist))
row_cols2 = pd.DataFrame(cats, index=to_cluster3.columns, columns=['venue']).venue.map(cat2col)
for c in clist:
    print([255*n for n in c])
clist

In [ ]:
# Draw the full plot
cnorm = TwoSlopeNorm(vmin=-1., vcenter=0., vmax=1.)
g = sns.clustermap(to_cluster3.corr(), center=0, cmap="vlag_r", norm=cnorm,
                   method='ward',# 'average','single','complete','weighted','centroid','median','ward'
                   metric='euclidean',# 'euclidean','cityblock','cosine','correlation','hamming','jaccard','chebyshev','canberra','braycurtis','matching'
                   row_colors=row_cols2, col_colors=row_cols2,
                   dendrogram_ratio=(.1, .2),
                   cbar_pos=(.7,.475,.02,.2),#(.775, .425, .02, .2),#cbar_pos=(.02, .32, .03, .2),
                   linewidths=0., figsize=(4.6,5),#(7,7.55),# square=True, 
                  )

#g.ax_row_dendrogram.remove()
g.ax_col_dendrogram.remove()
g.ax_heatmap.set_xticks([])
g.ax_heatmap.set_yticks([])
g.ax_heatmap.set_xlabel('OSM map feature')
g.ax_heatmap.set_ylabel('OSM map feature')
g.ax_row_colors.set_xticks([])
g.ax_col_colors.set_yticks([])
g.cax.set_ylim([-1,1])
g.cax.set_yticks([-1,-.5,0,.5,1])

# Move and reorient colorbar
g.cax.set_visible(False)  # hide original
cbar_ax = g.fig.add_axes([.275, .25, .2, .02])  # new horizontal axes
cb = plt.colorbar(g.ax_heatmap.collections[0], cax=cbar_ax, orientation='horizontal')
cb.outline.set_visible(False)
cb.set_ticks([-1, -0.5, 0, 0.5, 1])
cb.ax.set_xticklabels(cb.ax.get_xticklabels(), rotation=45)

plt.savefig(f'plots/fig6/contacts_nationwide_poi_clustering.jpg', bbox_inches='tight', dpi=300)
plt.savefig(f'plots/fig6/contacts_nationwide_poi_clustering.pdf', bbox_inches='tight')
plt.show()

In [ ]:
changes = pd.DataFrame()
for cl_h, c in zip(range(1,1+depth), sns.color_palette(None, depth)):
    nom = to_cluster2[cats_clus[cats_clus.cluster==cl_h].venue.tolist()].sum(axis=1)
    denom = base[cats_clus[cats_clus.cluster==cl_h].venue.tolist()].sum().sum()
    frac = nom.div(denom)
    print(f'cluster {cl_h}: {frac.sum()}')
    changes_tmp = frac.reset_index().rename(columns={0:'change_rel'})
    changes_tmp['cluster'] = cl_h
    changes = pd.concat([changes, changes_tmp])
changes

In [ ]:
clus2label = {cl_h: cl_lab for cl_h, cl_lab in zip(range(1,depth+1), cl_labels)}
for_heatmap7 = for_heatmap.merge(cats_clus, on='venue').groupby(['cluster','day','hour']).pair.sum().reset_index()
for_heatmap7['cluster_label'] = for_heatmap7.cluster.map(clus2label)
for_heatmap7

In [ ]:
sns.set_theme(style="ticks")

# Function to plot a heatmap inside each facet
def heatmap_func(data, **kwargs):
    pivoted = data.pivot(index="hour", columns="day", values="pair")  # Reshape
    sns.heatmap(pivoted.loc[::-1], cmap="coolwarm", cbar=False, robust=True, square=True, **kwargs)

# Create a FacetGrid, grouping by 'category'
g = sns.FacetGrid(for_heatmap7.reset_index().fillna(0), row="cluster_label", margin_titles=True, height=2.5, aspect=3.6,
                 )#row_order=mtypes_big)#['dining','outing','fanzone','shopping'])

# Map the custom heatmap function to each facet
g.map_dataframe(heatmap_func)

# Set yticks for all subplots
g.set(yticks=[24-x-.5 for x in range(0,24,2)], xticks=[x+.5 for x in range(len(set(data.day)))])
g.set_yticklabels(list(range(0,24,2)), rotation=0)
g.set_xticklabels([str(d.month).zfill(2)+'/'+str(d.day).zfill(2) if d.weekday()==6 else '' for d in sorted(set(data.day))], rotation=0)

for ax in g.axes.flat:
    #ax.set_title(ax.get_title().split('=')[1][1:])
    ax.tick_params(axis="x", labelbottom=True)

# Modify margin titles
g.set_titles(row_template="{row_name}")#, col_template="{col_name}", size=14, fontweight='bold')

g.tight_layout()
plt.savefig(f'plots/figs12/contacts_nationwide_poi_clustering2.jpg', bbox_inches='tight', dpi=300)
plt.savefig(f'plots/figs12/contacts_nationwide_poi_clustering2.pdf', bbox_inches='tight')
plt.show()

In [ ]:
for_andrzej = for_heatmap7.copy(deep=True)
for_andrzej['pair'] = for_andrzej.pair.astype(int)
#for_andrzej.to_csv('output/07_poi_contacts_clustered_for_andrzej.csv', index=False)
for_andrzej

In [ ]:
3*98*24

In [ ]:
dist_base8 = for_heatmap7[(for_heatmap7.day>=date(2024,6,10)) & (for_heatmap7.day!=date(2024,7,5))].copy(deep=True)
dist_base8['wd'] = [d.weekday() for d in dist_base8.day]
dist_base9 = dist_base8.copy(deep=True)
dist_base8 = dist_base8.fillna(0).groupby(['cluster','wd','hour']).pair.mean().reset_index().rename(columns={'pair':'pair_avg'})
#dist_base9 = dist_base9.fillna(0).groupby(['cluster','wd','hour']).pair.std().reset_index().rename(columns={'pair':'pair_std'})
dist_base8

In [ ]:
for_heatmap8 = for_heatmap7.copy(deep=True)
for_heatmap8['wd'] = [d.weekday() for d in for_heatmap8.day]
for_heatmap8 = for_heatmap8.merge(dist_base8, on=['cluster','wd','hour']).fillna(0)
for_heatmap8['diff'] = (for_heatmap8.pair - for_heatmap8.pair_avg)
for_heatmap8

In [ ]:
sns.set_theme(style="ticks")

# Function to plot a heatmap inside each facet
def heatmap_func(data, **kwargs):
    pivoted = data.pivot(index="hour", columns="day", values="diff")  # Reshape
    sns.heatmap(pivoted.loc[::-1], cmap="coolwarm", cbar=False, robust=True, square=True, center=0, **kwargs)

# Create a FacetGrid, grouping by 'category'
g = sns.FacetGrid(for_heatmap8.reset_index().fillna(0), row="cluster_label", margin_titles=True, height=2.5, aspect=3.6,
                 )#row_order=mtypes_big)#['dining','outing','fanzone','shopping'])

# Map the custom heatmap function to each facet
g.map_dataframe(heatmap_func)

# Set yticks for all subplots
g.set(yticks=[24-x-.5 for x in range(0,24,2)], xticks=[x+.5 for x in range(len(set(data.day)))])
g.set_yticklabels(list(range(0,24,2)), rotation=0)
g.set_xticklabels([str(d.month).zfill(2)+'/'+str(d.day).zfill(2) if d.weekday()==6 else '' for d in sorted(set(data.day))], rotation=0)

for ax in g.axes.flat:
    #ax.set_title(ax.get_title().split('=')[1][1:])
    ax.tick_params(axis="x", labelbottom=True)

# Modify margin titles
g.set_titles(row_template="{row_name}")#, col_template="{col_name}", size=14, fontweight='bold')

g.tight_layout()
plt.show()

In [ ]:
norm_h = for_heatmap8[(for_heatmap8.day>=date(2024,6,10)) & (~for_heatmap8.day.isin(dheres))].groupby(['cluster_label','wd','hour'])\
            .pair.mean().reset_index().rename(columns={'pair':'pair_avgsum'}).groupby(['cluster_label','wd']).pair_avgsum.sum().reset_index()
for_heatmap9 = for_heatmap8.merge(norm_h, on=['cluster_label','wd'])
for_heatmap9['rel_diff'] = for_heatmap9['diff'] / for_heatmap9.pair_avgsum
for_heatmap9 = for_heatmap9[(for_heatmap9.day >= date(2024,6,10))]# & (for_heatmap9.day <= date(2024,7,14))]
for_heatmap9

In [ ]:
base[cats_clus[cats_clus.cluster==3].venue.tolist()].sum().sum()

In [ ]:
norm_h[(norm_h.wd==4) & (norm_h.cluster_label=='routine')]

In [ ]:
here[(cats_clus[cats_clus.cluster==3].venue.tolist())].sum(axis=1)

In [ ]:
base[(cats_clus[cats_clus.cluster==3].venue.tolist())].sum(axis=1)

In [ ]:
to_cluster2[(cats_clus[cats_clus.cluster==3].venue.tolist())].sum(axis=1)

In [ ]:
for_heatmap8[(for_heatmap8.day==date(2024,7,5)) & (for_heatmap8.hour==19)]

In [ ]:
sns.set_theme(style="ticks")

# Function to plot a heatmap inside each facet
def heatmap_func(data, **kwargs):
    pivoted = data.pivot(index="hour", columns="day", values="rel_diff")  # Reshape
    sns.heatmap(pivoted.loc[::-1], cmap="coolwarm", cbar=True, robust=False, square=True, center=0,
                vmin=.9*for_heatmap9.rel_diff.min(), vmax=.9*for_heatmap9.rel_diff.max(),
                **kwargs)

# Create a FacetGrid, grouping by 'category'
g = sns.FacetGrid(for_heatmap9.reset_index().fillna(0), row="cluster_label", margin_titles=True, height=2.5, aspect=3.6,
                 )#row_order=mtypes_big)#['dining','outing','fanzone','shopping'])

# Map the custom heatmap function to each facet
g.map_dataframe(heatmap_func)

# Set yticks for all subplots
g.set(yticks=[24-x-.5 for x in range(0,24,2)], xticks=[x+.5 for x in range(len(set(for_heatmap9.day)))])
g.set_yticklabels(list(range(0,24,2)), rotation=0)
g.set_xticklabels([str(d.month).zfill(2)+'/'+str(d.day).zfill(2) if d.weekday()==6 else '' for d in sorted(set(for_heatmap9.day))], rotation=0)

for ax in g.axes.flat:
    #ax.set_title(ax.get_title().split('=')[1][1:])
    ax.tick_params(axis="x", labelbottom=True)

# Modify margin titles
g.set_titles(row_template="{row_name}")#, col_template="{col_name}", size=14, fontweight='bold')

g.tight_layout()
plt.show()

In [ ]:
for_heatmap9[for_heatmap9.day==date(2024,7,5)].groupby(['cluster_label','day']).rel_diff.sum()#.apply(len)

In [ ]:
fig, ax = plt.subplots(figsize=[10,4])

# Plot the responses for different events and regions
g = sns.lineplot(data=changes, x="hour", y="change_rel", hue="cluster",# style="event",
                 marker="o", palette=clist, markeredgecolor='none',
                )
g = sns.lineplot(data=for_heatmap9[(for_heatmap9.day!=date(2024,7,5)) & (for_heatmap9.wd==4)], x="hour", y="rel_diff",
                 hue="cluster", palette=clist, # style="event",
                 marker="o", markeredgecolor='none', legend=False, errorbar=('ci',95),
                 lw=0, ms=0, err_style='band',
                )
ax.plot([0,23], [0,0], c='k', alpha=.25)
yu, yo = changes.change_rel.min()-.005, changes.change_rel.max()+.01
ax.fill_between([16,18], [yu,yu], [yo,yo], color='gray', zorder=0, alpha=.25)
ax.fill_between([18,20], [yu,yu], [yo,yo], color='gray', zorder=0, alpha=.5)
ax.fill_between([20,22], [yu,yu], [yo,yo], color='gray', zorder=0, alpha=.25)
ax.text(17, yo-.005, 'before', va='center', ha='center')#, fontsize=8)
ax.text(19, yo-.005, 'during', va='center', ha='center')#, fontsize=8)
ax.text(21, yo-.005, 'after', va='center', ha='center')#, fontsize=8)

g.set(ylabel='share of daily contacts')
g.set(xticks=range(24), ylim=[yu,yo])
sns.despine()

# Get the legend object
ax.legend(loc='upper left', title='cluster')
legend = ax.get_legend()
for text, new_label in zip(legend.texts, cl_labels):
    text.set_text(new_label)

plt.savefig(f'plots/fig6/contacts_nationwide_poi_clustering3.jpg', bbox_inches='tight', dpi=300)
plt.savefig(f'plots/fig6/contacts_nationwide_poi_clustering3.pdf', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=[7,1.4*depth], nrows=depth)

for r in range(depth):
    # Plot the responses for different events and regions
    changes_r = changes[changes.cluster==r+1]
    for_heatmap9_r = for_heatmap9[(for_heatmap9.day!=date(2024,7,5)) & (for_heatmap9.wd==4) & (for_heatmap9.cluster==r+1)]
    g = sns.lineplot(ax=ax[r], data=changes_r, x="hour", y="change_rel", hue="cluster",# style="event",
                     marker="o", palette=[clist[r]], markeredgecolor='none', legend=False,
                    )
    g = sns.lineplot(ax=ax[r], data=for_heatmap9_r, x="hour", y="rel_diff",
                     hue="cluster", palette=[clist[r]], # style="event",
                     marker="o", markeredgecolor='none', legend=False, errorbar=('ci',95),
                     lw=0, ms=0, err_style='band',
                    )
    ax[r].plot([0,23], [0,0], c='k', alpha=.25)
    yu, yo = changes.change_rel.min()-.005, changes.change_rel.max()+.0175
    ax[r].fill_between([16,18], [yu,yu], [yo,yo], color='gray', zorder=0, alpha=.25)
    ax[r].fill_between([18,20], [yu,yu], [yo,yo], color='gray', zorder=0, alpha=.5)
    ax[r].fill_between([20,22], [yu,yu], [yo,yo], color='gray', zorder=0, alpha=.25)
    if r==0:
        ax[r].text(17, yo-.005, 'before', va='center', ha='center')#, fontsize=8)
        ax[r].text(19, yo-.02, 'during', va='center', ha='center')#, fontsize=8)
        ax[r].text(21, yo-.005, 'after', va='center', ha='center')#, fontsize=8)

    if r==1:
        g.set(ylabel='share of daily contacts')
    else:
        g.set(ylabel='')
    g.set(xticks=range(24), ylim=[yu,yo])
    if r<depth-1:
        g.set(xlabel='', xticklabels=[])
    sns.despine()
    
    # Get the legend object
    ax[r].text(0, .04, cl_labels[r], ha='left', va='center')
    #ax[r].legend(loc='upper left', title='cluster')
    #legend = ax[r].get_legend()
    #for text, new_label in zip(legend.texts, cl_labels):
    #    text.set_text(new_label)

plt.savefig(f'plots/fig7/contacts_nationwide_poi_clustering3.jpg', bbox_inches='tight', dpi=300)
plt.savefig(f'plots/fig7/contacts_nationwide_poi_clustering3.pdf', bbox_inches='tight')
plt.show()

In [ ]:
for_heatmap9[for_heatmap9.day!=date(2024,7,5)].groupby(['cluster','wd']).rel_diff.sum().reset_index()

In [ ]:
# Example
value = changes[(changes.cluster==1) & (changes.hour==17)].change_rel.iloc[0]
distribution = for_heatmap9[(for_heatmap9.day!=date(2024,7,5)) & (for_heatmap9.wd==4) & (for_heatmap9.cluster==1) & (for_heatmap9.hour==17)].rel_diff.tolist()
 
# Two-sided empirical p-value
p_empirical = (np.sum(np.abs(distribution - np.median(distribution)) >= np.abs(value - np.median(distribution))) + 1) / (len(distribution) + 1)
print(f"Empirical two-sided p-value: {p_empirical:.4f}")